<a href="https://colab.research.google.com/github/maruwagner893-droid/cloud-provider-analytics/blob/main/notebooks/02_pipeline_etapa2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
!pip -q install pyspark

In [44]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("cloud-provider-analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("Versión de Spark:", spark.version)

Versión de Spark: 4.0.5


In [45]:
import os

REPO = "/content/cloud-provider-analytics"

if not os.path.isdir(REPO):
    !git clone https://github.com/maruwagner893-droid/cloud-provider-analytics {REPO}

!unzip -q -o {REPO}/data/cloud_provider_challenge_dataset_v1.zip -d {REPO}/data/extracted

In [46]:
LANDING = os.path.join(REPO, "data/extracted/datalake/landing")
print("Landing:", LANDING)
print(sorted(os.listdir(LANDING)))

Landing: /content/cloud-provider-analytics/data/extracted/datalake/landing
['billing_monthly.csv', 'customers_orgs.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'resources.csv', 'support_tickets.csv', 'usage_events_stream', 'users.csv']


In [47]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, DateType, DoubleType
)

schema_customers = StructType([
    StructField("org_id",           StringType(),  True),
    StructField("org_name",         StringType(),  True),
    StructField("industry",         StringType(),  True),
    StructField("hq_region",        StringType(),  True),
    StructField("plan_tier",        StringType(),  True),
    StructField("is_enterprise",    BooleanType(), True),
    StructField("signup_date",      DateType(),    True),
    StructField("sales_rep",        StringType(),  True),
    StructField("lifecycle_stage",  StringType(),  True),
    StructField("marketing_source", StringType(),  True),
    StructField("nps_score",        DoubleType(),  True),
])

customers = (
    spark.read
    .option("header", True)
    .schema(schema_customers)
    .csv(os.path.join(LANDING, "customers_orgs.csv"))
)

customers.printSchema()
customers.show(5, truncate=False)

# Verificación
print("Filas:", customers.count())                                                 # 80
print("signup_date nulos:", customers.filter(F.col("signup_date").isNull()).count())  # 0
print("nps_score nulos:", customers.filter(F.col("nps_score").isNull()).count())      # 11

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)

+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0  |Education    |sa-east   |standard |true         |2025-05-26 |rep_

In [48]:
BRONZE = os.path.join(REPO, "data/bronze")

customers_bronze = (
    customers
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
)

(
    customers_bronze
    .coalesce(1)                      # 80 filas: un solo archivo, sin archivos chicos
    .write.mode("overwrite")          # re-ejecutable: pisa el resultado anterior
    .partitionBy("plan_tier")
    .parquet(os.path.join(BRONZE, "customers_orgs"))
)

# Verificación: leer lo que se guardó
check = spark.read.parquet(os.path.join(BRONZE, "customers_orgs"))
check.printSchema()
check.select("org_id", "ingest_ts", "source_file").show(3, truncate=False)
print("Filas en Bronze:", check.count())   # 80

!ls -la {BRONZE}/customers_orgs

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- plan_tier: string (nullable = true)

+------------+--------------------------+------------------+
|org_id      |ingest_ts                 |source_file       |
+------------+--------------------------+------------------+
|org_xaji0y6d|2026-11-17 15:10:48.147834|customers_orgs.csv|
|org_pbhsahxt|2026-11-17 15:10:48.147834|customers_orgs.csv|
|org_8mdd4v30|2026-11-17 15:10:48.147834|customers_orgs.csv|
+------------+--------------------------+------------------+
only showing top

In [49]:
schema_users = StructType([
    StructField("user_id",    StringType(),  True),
    StructField("org_id",     StringType(),  True),
    StructField("email",      StringType(),  True),
    StructField("role",       StringType(),  True),
    StructField("active",     BooleanType(), True),
    StructField("created_at", DateType(),    True),
    StructField("last_login", DateType(),    True),
])

schema_billing = StructType([
    StructField("invoice_id",           StringType(), True),
    StructField("org_id",               StringType(), True),
    StructField("month",                DateType(),   True),
    StructField("subtotal",             DoubleType(), True),
    StructField("credits",              DoubleType(), True),
    StructField("taxes",                DoubleType(), True),
    StructField("currency",             StringType(), True),
    StructField("exchange_rate_to_usd", DoubleType(), True),
])

def ingest_master(name, schema, key, part_col):
    """Landing (CSV) -> Bronze (Parquet): tipifica, deduplica por clave y agrega metadatos."""
    df = (
        spark.read.option("header", True).schema(schema)
        .csv(os.path.join(LANDING, f"{name}.csv"))
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_name"))
    )
    n_in = df.count()
    df = df.dropDuplicates([key])
    n_out = df.count()
    out = os.path.join(BRONZE, name)
    df.coalesce(1).write.mode("overwrite").partitionBy(part_col).parquet(out)
    print(f"{name}: leídas={n_in}  duplicadas quitadas={n_in - n_out}  guardadas={n_out}  -> {out}")

ingest_master("customers_orgs",  schema_customers, "org_id",     "plan_tier")
ingest_master("users",           schema_users,     "user_id",    "role")
ingest_master("billing_monthly", schema_billing,   "invoice_id", "month")
# Verificación
users_b   = spark.read.parquet(os.path.join(BRONZE, "users"))
billing_b = spark.read.parquet(os.path.join(BRONZE, "billing_monthly"))
print("users last_login nulos:", users_b.filter(F.col("last_login").isNull()).count())      # 139
print("billing credits nulos:", billing_b.filter(F.col("credits").isNull()).count())        # 137
print("billing subtotal < 0:", billing_b.filter(F.col("subtotal") < 0).count())             # 13

customers_orgs: leídas=80  duplicadas quitadas=0  guardadas=80  -> /content/cloud-provider-analytics/data/bronze/customers_orgs
users: leídas=800  duplicadas quitadas=0  guardadas=800  -> /content/cloud-provider-analytics/data/bronze/users
billing_monthly: leídas=240  duplicadas quitadas=0  guardadas=240  -> /content/cloud-provider-analytics/data/bronze/billing_monthly
users last_login nulos: 139
billing credits nulos: 137
billing subtotal < 0: 13


In [50]:
for m in ["customers_orgs", "users", "billing_monthly"]:
    print("==", m)
    for ruta, _, archivos in sorted(os.walk(os.path.join(BRONZE, m))):
        for a in sorted(archivos):
            if a.endswith(".parquet"):
                p = os.path.join(ruta, a)
                print(os.path.relpath(p, BRONZE), os.path.getsize(p), "bytes")

== customers_orgs
customers_orgs/plan_tier=enterprise/part-00000-4d1491f2-7009-4e32-ae03-20991d2b2193.c000.snappy.parquet 4437 bytes
customers_orgs/plan_tier=free/part-00000-4d1491f2-7009-4e32-ae03-20991d2b2193.c000.snappy.parquet 4354 bytes
customers_orgs/plan_tier=pro/part-00000-4d1491f2-7009-4e32-ae03-20991d2b2193.c000.snappy.parquet 4868 bytes
customers_orgs/plan_tier=standard/part-00000-4d1491f2-7009-4e32-ae03-20991d2b2193.c000.snappy.parquet 5442 bytes
== users
users/role=admin/part-00000-60f7d945-6ee2-4b0e-9189-4a679847196c.c000.snappy.parquet 5847 bytes
users/role=analyst/part-00000-60f7d945-6ee2-4b0e-9189-4a679847196c.c000.snappy.parquet 6609 bytes
users/role=data_engineer/part-00000-60f7d945-6ee2-4b0e-9189-4a679847196c.c000.snappy.parquet 9007 bytes
users/role=developer/part-00000-60f7d945-6ee2-4b0e-9189-4a679847196c.c000.snappy.parquet 7073 bytes
users/role=devops/part-00000-60f7d945-6ee2-4b0e-9189-4a679847196c.c000.snappy.parquet 7935 bytes
users/role=ml_engineer/part-00000

In [51]:
from pyspark.sql.types import TimestampType, IntegerType, LongType
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

schema_events = StructType([
    StructField("event_id",           StringType(),    True),
    StructField("timestamp",          TimestampType(), True),
    StructField("org_id",             StringType(),    True),
    StructField("resource_id",        StringType(),    True),
    StructField("service",            StringType(),    True),
    StructField("region",             StringType(),    True),
    StructField("metric",             StringType(),    True),
    StructField("value",              StringType(),    True),   # llega como número o como texto
    StructField("unit",               StringType(),    True),
    StructField("cost_usd_increment", DoubleType(),    True),
    StructField("schema_version",     IntegerType(),   True),
    StructField("carbon_kg",          DoubleType(),    True),   # solo en v2
    StructField("genai_tokens",       LongType(),      True),   # solo servicio genai
])

EVENTS_LANDING = os.path.join(LANDING, "usage_events_stream")
REPLAY = os.path.join(REPO, "data/replay/usage_events_stream")

events_all = spark.read.schema(schema_events).json(EVENTS_LANDING)

(
    events_all
    .repartitionByRange(120, "timestamp")      # 120 archivos, cada uno con un tramo de tiempo
    .sortWithinPartitions("timestamp")
    .write.mode("overwrite")
    .json(REPLAY)
)

# Verificación: los archivos tienen que quedar ordenados y sin solaparse
replay = (
    spark.read.schema(schema_events).json(REPLAY)
    .withColumn("f", F.col("_metadata.file_name"))
)
rangos = (
    replay.groupBy("f")
    .agg(F.count("*").alias("n"), F.min("timestamp").alias("desde"), F.max("timestamp").alias("hasta"))
    .orderBy("f")
)
rangos.show(5, truncate=False)

w = Window.orderBy("f")
solapes = (
    rangos.withColumn("desde_sig", F.lead("desde").over(w))
    .filter(F.col("hasta") > F.col("desde_sig"))
    .count()
)
print("Archivos:", rangos.count())
print("Eventos totales:", replay.count())                 # 43200
print("Archivos que se solapan con el siguiente:", solapes)   # 0

+---------------------------------------------------------+---+-------------------+-------------------+
|f                                                        |n  |desde              |hasta              |
+---------------------------------------------------------+---+-------------------+-------------------+
|part-00000-234aec21-7d6f-4716-b91b-ece866e238fd-c000.json|354|2025-07-03 00:02:00|2025-07-03 12:41:00|
|part-00001-234aec21-7d6f-4716-b91b-ece866e238fd-c000.json|362|2025-07-03 12:42:00|2025-07-03 23:50:00|
|part-00002-234aec21-7d6f-4716-b91b-ece866e238fd-c000.json|359|2025-07-03 23:54:00|2025-07-04 12:05:00|
|part-00003-234aec21-7d6f-4716-b91b-ece866e238fd-c000.json|356|2025-07-04 12:08:00|2025-07-04 23:40:00|
|part-00004-234aec21-7d6f-4716-b91b-ece866e238fd-c000.json|371|2025-07-04 23:43:00|2025-07-05 11:31:00|
+---------------------------------------------------------+---+-------------------+-------------------+
only showing top 5 rows
Archivos: 120
Eventos totales: 43200
Arc

In [52]:
BRONZE_EVENTS = os.path.join(BRONZE, "usage_events_stream")
CHECKPOINT    = os.path.join(REPO, "data/checkpoints/bronze_events")

stream = (
    spark.readStream
    .schema(schema_events)
    .option("maxFilesPerTrigger", 10)         # 10 archivos por micro-lote: menos archivos chicos
    .json(REPLAY)
)

bronze_stream = (
    stream
    .withColumn("value", F.col("value").cast("double"))        # texto numérico -> número
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("year", F.year("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withWatermark("timestamp", "10 minutes")                   # watermark de 10 minutos
    .dropDuplicatesWithinWatermark(["event_id"])                # deduplica por event_id
)

def run_bronze_events():
    q = (
        bronze_stream.writeStream
        .queryName("bronze_events")
        .format("parquet")
        .outputMode("append")
        .option("path", BRONZE_EVENTS)
        .option("checkpointLocation", CHECKPOINT)
        .partitionBy("year", "month")                           # una carpeta por mes
        .trigger(availableNow=True)                             # procesa todo lo pendiente y termina
        .start()
    )
    q.awaitTermination()
    return q

def contar_bronze():
    d = spark.read.parquet(BRONZE_EVENTS)
    print("Eventos en Bronze:", d.count())
    d.groupBy("year", "month").count().orderBy("year", "month").show()

# Monitoreo de cada micro-lote con StreamingQueryListener (Clase 03)
from pyspark.sql.streaming import StreamingQueryListener
import time

lotes = []   # métricas de cada micro-lote

class Monitor(StreamingQueryListener):
    def onQueryStarted(self, event):
        print("inicio", event.id, event.name)
    def onQueryProgress(self, event):
        p = event.progress
        lotes.append((p.batchId, p.numInputRows, p.inputRowsPerSecond, p.processedRowsPerSecond))
    def onQueryIdle(self, event):
        pass
    def onQueryTerminated(self, event):
        print("fin", event.id, event.exception)

monitor = Monitor()
spark.streams.addListener(monitor)

# Primera ejecución
consulta = run_bronze_events()
time.sleep(2)                          # los eventos del listener llegan con un pequeño retraso
spark.streams.removeListener(monitor)  # el monitor solo mira esta ejecución

for batch_id, filas, entrada, procesadas in lotes:
    print(batch_id, filas, entrada, procesadas)
print("Micro-lotes con datos:", sum(1 for l in lotes if l[1] > 0))   # 12
print("Filas leídas en total:", sum(l[1] for l in lotes))             # 43200

contar_bronze()

inicio 3d16a07f-8973-43e2-965c-046d4ceff8a8 bronze_events
fin 3d16a07f-8973-43e2-965c-046d4ceff8a8 None
26 3603 0.0 4393.90243902439
27 3609 4390.51094890511 3269.0217391304345
28 3563 3207.020702070207 3538.2323733862963
29 3593 3560.951437066403 4197.429906542056
30 3603 4194.4121071012805 2746.189024390244
31 3609 2744.4866920152094 2732.0211960635884
32 3605 2720.7547169811323 2283.0905636478783
33 3578 2260.2653190145293 2750.1921598770177
34 3575 2743.6684574059864 2976.686094920899
35 3604 2995.843724023275 3900.4329004329
36 3599 3882.4163969795036 4465.260545905707
37 3659 4522.867737948084 4376.794258373206
Micro-lotes con datos: 12
Filas leídas en total: 43200
Eventos en Bronze: 43200
+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2025|    7|20880|
|2025|    8|22320|
+----+-----+-----+



In [53]:
import shutil, glob

def total_bronze():
    return spark.read.parquet(BRONZE_EVENTS).count()

# A) Re-ejecutar sin archivos nuevos: el checkpoint evita reprocesar
antes = total_bronze()
run_bronze_events()
despues = total_bronze()
print(f"A) Re-ejecución sin archivos nuevos -> antes={antes}  después={despues}")

# B) Llega un archivo repetido (mismos event_id): no debe sumar eventos
ultimo = sorted(glob.glob(os.path.join(REPLAY, "part-*.json")))[-1]
repetido = os.path.join(REPLAY, "part-99999-repetido.json")
shutil.copy(ultimo, repetido)

antes = total_bronze()
run_bronze_events()
despues = total_bronze()
print(f"B) Llega un archivo repetido      -> antes={antes}  después={despues}")

os.remove(repetido)   # dejamos el replay como estaba

print("Checkpoint:", os.listdir(CHECKPOINT))

A) Re-ejecución sin archivos nuevos -> antes=43200  después=43200
B) Llega un archivo repetido      -> antes=43200  después=43200
Checkpoint: ['state', 'metadata', 'offsets', 'commits', '.metadata.crc', 'sources']


In [54]:
SILVER_EVENTS     = os.path.join(REPO, "data/silver/usage_events")
QUARANTINE_EVENTS = os.path.join(REPO, "data/quarantine/usage_events")
CHECKPOINT_SILVER = os.path.join(REPO, "data/checkpoints/silver_events")

customers_dim = (
    spark.read.parquet(os.path.join(BRONZE, "customers_orgs"))
    .select("org_id", "industry", F.col("hq_region").alias("org_hq_region"),
            "plan_tier", "is_enterprise")
)

unidad_por_metrica = (
    F.when(F.col("metric") == "cpu_hours", "hours")
     .when(F.col("metric") == "requests", "count")
     .when(F.col("metric") == "storage_gb_hours", "gb_hours")
)

def procesar_lote(batch_df, batch_id):
    motivo = (
        F.when(F.col("event_id").isNull(), "event_id_nulo")
         .when(F.col("cost_usd_increment") < -0.01, "cost_usd_increment_menor_a_-0.01")
    )
    marcado = batch_df.withColumn("quarantine_reason", motivo)

    invalidos = (
        marcado.filter(F.col("quarantine_reason").isNotNull())
        .withColumn("quarantine_ts", F.current_timestamp())
    )
    validos = marcado.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason")

    silver = (
        validos
        .withColumn("unit_imputed", F.col("unit").isNull())
        .withColumn("unit", F.coalesce(F.col("unit"), unidad_por_metrica))
        .withColumn("value_missing", F.col("value").isNull())
        .withColumn("usage_date", F.to_date("timestamp"))
        .withColumn("requests",         F.when(F.col("metric") == "requests",         F.col("value")))
        .withColumn("cpu_hours",        F.when(F.col("metric") == "cpu_hours",        F.col("value")))
        .withColumn("storage_gb_hours", F.when(F.col("metric") == "storage_gb_hours", F.col("value")))
        .join(F.broadcast(customers_dim), on="org_id", how="left")
    )

    silver.coalesce(1).write.mode("append").partitionBy("year", "month").parquet(SILVER_EVENTS)
    if invalidos.count() > 0:                                   # solo escribe si hay inválidos
        invalidos.coalesce(1).write.mode("append").parquet(QUARANTINE_EVENTS)
bronze_schema = spark.read.parquet(BRONZE_EVENTS).schema
silver_stream = (
    spark.readStream.schema(bronze_schema)
    .option("maxFilesPerTrigger", 10)
    .parquet(BRONZE_EVENTS)
)

def run_silver_events():
    q = (
        silver_stream.writeStream
        .foreachBatch(procesar_lote)
        .option("checkpointLocation", CHECKPOINT_SILVER)
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()

run_silver_events()

# Verificación
s = spark.read.parquet(SILVER_EVENTS)
q = spark.read.parquet(QUARANTINE_EVENTS)
n_s, n_q = s.count(), q.count()
print("Silver:", n_s)                                              # 42989
print("Quarantine:", n_q)                                          # 211
print("% inválidos:", round(n_q / (n_s + n_q) * 100, 2))           # 0.49
s.groupBy("year", "month").count().orderBy("year", "month").show()  # 20778 / 22211
print("unit nulos en Silver:", s.filter(F.col("unit").isNull()).count())          # 0
print("unit imputada:", s.filter("unit_imputed").count())                         # 2069
print("value nulo:", s.filter("value_missing").count())                           # 873
q.groupBy("quarantine_reason").count().show(truncate=False)
q.select("event_id", "cost_usd_increment", "quarantine_reason", "source_file") \
 .orderBy("cost_usd_increment").show(5, truncate=False)                           # mínimo -154.46
s.select("event_id", "service", "metric", "unit", "requests", "cpu_hours",
         "storage_gb_hours", "industry", "plan_tier").show(5, truncate=False)

Silver: 42989
Quarantine: 211
% inválidos: 0.49
+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2025|    7|20778|
|2025|    8|22211|
+----+-----+-----+

unit nulos en Silver: 0
unit imputada: 2069
value nulo: 873
+--------------------------------+-----+
|quarantine_reason               |count|
+--------------------------------+-----+
|cost_usd_increment_menor_a_-0.01|211  |
+--------------------------------+-----+

+----------------+------------------+--------------------------------+---------------------------------------------------------+
|event_id        |cost_usd_increment|quarantine_reason               |source_file                                              |
+----------------+------------------+--------------------------------+---------------------------------------------------------+
|evt_t5brta67j5l3|-154.4608         |cost_usd_increment_menor_a_-0.01|part-00105-fcdb1dbe-7c97-4b11-89ea-4a05eb4ab6fa-c000.json|
|evt_9ur456xoexl5|-19.8245          |cost_usd_increment

In [55]:
# Silver de billing_monthly: créditos, moneda normalizada a USD y regla de calidad
SILVER_BILLING     = os.path.join(REPO, "data/silver/billing_monthly")
QUARANTINE_BILLING = os.path.join(REPO, "data/quarantine/billing_monthly")

billing_bronze = spark.read.parquet(os.path.join(BRONZE, "billing_monthly"))

tasa_valida = F.coalesce(F.col("exchange_rate_to_usd") > 0, F.lit(False))

billing_marcado = (
    billing_bronze
    .withColumn("credits_imputed", F.col("credits").isNull())           # marca cuáles se completaron
    .withColumn("credits", F.coalesce(F.col("credits"), F.lit(0.0)))    # credits nulo = 0
    .withColumn("is_credit", F.col("subtotal") < 0)                     # las 13 facturas negativas
    .withColumn("net_usd", F.round(
        (F.col("subtotal") - F.col("credits")) * F.col("exchange_rate_to_usd"), 2))
    .withColumn("quarantine_reason",
        F.when(~tasa_valida, "exchange_rate_nula_o_no_positiva"))
)

billing_validos   = billing_marcado.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason")
billing_invalidos = (billing_marcado.filter(F.col("quarantine_reason").isNotNull())
                     .withColumn("quarantine_ts", F.current_timestamp()))

# overwrite: re-ejecutable, pisa el resultado anterior
billing_validos.coalesce(1).write.mode("overwrite").partitionBy("month").parquet(SILVER_BILLING)
billing_invalidos.coalesce(1).write.mode("overwrite").parquet(QUARANTINE_BILLING)

# Verificación
s = spark.read.parquet(SILVER_BILLING)
print("Filas en Silver:", s.count())                                        # 240
print("Quarantine:", billing_invalidos.count())                             # 0
print("credits imputados a 0:", s.filter("credits_imputed").count())        # 137
print("Facturas con crédito (subtotal < 0):", s.filter("is_credit").count())  # 13
s.groupBy("currency").count().orderBy("currency").show()                    # ARS 51 · EUR 29 · USD 160
total_org = s.filter(F.col("org_id") == "org_xaji0y6d").agg(F.sum("net_usd")).first()[0]
print("Total org_xaji0y6d (USD):", round(total_org, 2))                     # 1729.18, igual que la sección 9

Filas en Silver: 240
Quarantine: 0
credits imputados a 0: 137
Facturas con crédito (subtotal < 0): 13
+--------+-----+
|currency|count|
+--------+-----+
|     ARS|   51|
|     EUR|   29|
|     USD|  160|
+--------+-----+

Total org_xaji0y6d (USD): 1729.18


In [56]:
GOLD_USAGE = os.path.join(REPO, "data/gold/org_daily_usage_by_service")

silver = spark.read.parquet(SILVER_EVENTS)

org_daily_usage_by_service = (
    silver
    .groupBy("org_id", "usage_date", "service")
    .agg(
        F.round(F.sum("cost_usd_increment"), 4).alias("daily_cost_usd"),
        F.sum("requests").alias("requests"),
        F.sum("cpu_hours").alias("cpu_hours"),
        F.sum("storage_gb_hours").alias("storage_gb_hours"),
        F.sum("genai_tokens").alias("genai_tokens"),
        F.sum("carbon_kg").alias("carbon_kg"),
        F.count("*").alias("n_events"),
    )
)

# Evidencia del shuffle (captura para evidence/)
org_daily_usage_by_service.explain(mode="formatted")

(
    org_daily_usage_by_service
    .coalesce(1)
    .write.mode("overwrite")
    .parquet(GOLD_USAGE)
)

# Verificación
gold = spark.read.parquet(GOLD_USAGE)
print("Filas Gold:", gold.count())                                        # 11050
claves = gold.select("org_id", "usage_date", "service").distinct().count()
print("Claves únicas (grano):", claves)                                   # 11050
print("Suma n_events:", gold.agg(F.sum("n_events")).first()[0])           # 42989
print("Costo total:", round(gold.agg(F.sum("daily_cost_usd")).first()[0], 2))   # ~148351.58
print("Requests total:", gold.agg(F.sum("requests")).first()[0])          # 2281020.0
print("genai_tokens total:", gold.agg(F.sum("genai_tokens")).first()[0])  # 2545481
print("Filas con genai_tokens:", gold.filter(F.col("genai_tokens").isNotNull()).count())  # 848
gold.orderBy("org_id", "usage_date", "service").show(5, truncate=False)

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan parquet  (1)


(1) Scan parquet 
Output [11]: [org_id#60714, service#60718, cost_usd_increment#60723, carbon_kg#60725, genai_tokens#60726L, usage_date#60731, requests#60732, cpu_hours#60733, storage_gb_hours#60734, year#60739, month#60740]
Batched: true
Location: InMemoryFileIndex [file:/content/cloud-provider-analytics/data/silver/usage_events]
ReadSchema: struct<org_id:string,service:string,cost_usd_increment:double,carbon_kg:double,genai_tokens:bigint,usage_date:date,requests:double,cpu_hours:double,storage_gb_hours:double>

(2) Project
Output [9]: [org_id#60714, service#60718, cost_usd_increment#60723, carbon_kg#60725, genai_tokens#60726L, usage_date#60731, requests#60732, cpu_hours#60733, storage_gb_hours#60734]
Input [11]: [org_id#60714, service#60718, cost_usd_increment#60723, carbon_kg#60725, genai_tokens#60726L, usage_date#607

In [57]:
# Analítica: detección de anomalías de costo con el método IQR (visto en Minería de Datos I)

GOLD_ANOMALIAS = os.path.join(REPO, "data/gold/cost_anomaly_mart")

gold = spark.read.parquet(GOLD_USAGE)

cuartiles = (
    gold.groupBy("org_id", "service")
    .agg(
        F.percentile("daily_cost_usd", 0.25).alias("q1"),
        F.percentile("daily_cost_usd", 0.75).alias("q3"),
        F.count("*").alias("n_dias"),
    )
    .withColumn("iqr", F.col("q3") - F.col("q1"))
    .withColumn("sup", F.col("q3") + 1.5 * F.col("iqr"))
)

cost_anomaly_mart = (
    gold.select("org_id", "usage_date", "service", "daily_cost_usd")
    .join(F.broadcast(cuartiles), on=["org_id", "service"], how="left")
    .withColumn("is_anomaly", F.col("daily_cost_usd") > F.col("sup"))
    .withColumn("q1",  F.round("q1", 4))
    .withColumn("q3",  F.round("q3", 4))
    .withColumn("iqr", F.round("iqr", 4))
    .withColumn("sup", F.round("sup", 4))
)

cost_anomaly_mart.coalesce(1).write.mode("overwrite").parquet(GOLD_ANOMALIAS)

# Verificación
a = spark.read.parquet(GOLD_ANOMALIAS)
print("Filas:", a.count())                                       # 11050
print("Combinaciones org/servicio:", cuartiles.count())         # 262
print("Anomalías (costo > q3 + 1.5*iqr):", a.filter("is_anomaly").count())   # 196
a.filter("is_anomaly").orderBy(F.col("daily_cost_usd").desc()) \
 .select("org_id", "usage_date", "service", "daily_cost_usd", "q3", "sup") \
 .show(5, truncate=False)                                        # primero: org_rixa11dp, 2025-08-24, genai, 340.53 (sup 59.60)

Filas: 11050
Combinaciones org/servicio: 262
Anomalías (costo > q3 + 1.5*iqr): 196
+------------+----------+-------+--------------+-------+-------+
|org_id      |usage_date|service|daily_cost_usd|q3     |sup    |
+------------+----------+-------+--------------+-------+-------+
|org_rixa11dp|2025-08-24|genai  |340.5259      |34.473 |59.5995|
|org_c11ertj5|2025-07-11|genai  |334.6767      |30.7304|57.1145|
|org_tvhhpbmy|2025-07-04|genai  |313.4168      |30.2732|52.9081|
|org_x7eedtjv|2025-07-09|compute|311.255       |21.1819|37.0582|
|org_5935a0l7|2025-07-09|compute|271.6404      |26.8324|49.3971|
+------------+----------+-------+--------------+-------+-------+
only showing top 5 rows


In [58]:
def tam(ruta):
    total, n = 0, 0
    for carpeta, _, archivos in os.walk(ruta):
        for a in archivos:
            if a.endswith((".parquet", ".json", ".jsonl")):
                total += os.path.getsize(os.path.join(carpeta, a))
                n += 1
    return n, round(total / 1024 / 1024, 2)

zonas = {
    "Landing (eventos)": EVENTS_LANDING,
    "Bronze": BRONZE,
    "Silver": SILVER_EVENTS,
    "Gold": GOLD_USAGE,
    "Quarantine": QUARANTINE_EVENTS,
}

for nombre, ruta in zonas.items():
    n, mb = tam(ruta)
    print(f"{nombre:20s} {n:4d} archivos  {mb:8.2f} MB  {ruta}")

Landing (eventos)     120 archivos     12.33 MB  /content/cloud-provider-analytics/data/extracted/datalake/landing/usage_events_stream
Bronze                117 archivos      2.69 MB  /content/cloud-provider-analytics/data/bronze
Silver                 13 archivos      1.96 MB  /content/cloud-provider-analytics/data/silver/usage_events
Gold                    1 archivos      0.26 MB  /content/cloud-provider-analytics/data/gold/org_daily_usage_by_service
Quarantine             11 archivos      0.08 MB  /content/cloud-provider-analytics/data/quarantine/usage_events


In [59]:
TEST_LATE = os.path.join(REPO, "data/test_late")
shutil.rmtree(TEST_LATE, ignore_errors=True)       # empezar limpio

late_stream = (
    spark.readStream
    .schema(schema_events)
    .option("maxFilesPerTrigger", 1)               # un archivo por micro-lote
    .json(EVENTS_LANDING)                          # archivos ORIGINALES, sin ordenar
    .withWatermark("timestamp", "10 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
)

q = (
    late_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", os.path.join(TEST_LATE, "out"))
    .option("checkpointLocation", os.path.join(TEST_LATE, "checkpoint"))
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

total = 43200
pasaron = spark.read.parquet(os.path.join(TEST_LATE, "out")).count()
descartados = total - pasaron
print("Eventos de entrada:", total)
print("Pasaron el watermark:", pasaron)
print("Descartados por llegar tarde:", descartados)
print("% descartado:", round(descartados / total * 100, 2))

# Contraste: el replay ordenado (Bronze) no pierde nada
print("Bronze con replay ordenado:", spark.read.parquet(BRONZE_EVENTS).count())   # 43200

# Idempotencia de Silver: re-ejecutar no cambia el conteo
antes = spark.read.parquet(SILVER_EVENTS).count()
run_silver_events()
despues = spark.read.parquet(SILVER_EVENTS).count()
print(f"Silver re-ejecutado -> antes={antes}  después={despues}")        # 42989 / 42989

Eventos de entrada: 43200
Pasaron el watermark: 733
Descartados por llegar tarde: 42467
% descartado: 98.3
Bronze con replay ordenado: 43200
Silver re-ejecutado -> antes=42989  después=42989


In [60]:
# Prueba controlada de late data (misma lógica que la Demo 4 de clase: "Evento muy tardío")
import json, shutil
from pathlib import Path
from datetime import datetime, timedelta

tardio_input = Path(REPO) / "data/test_tardio/in"
tardio_checkpoint = Path(REPO) / "data/test_tardio/checkpoint"

for p in [tardio_input, tardio_checkpoint]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

def escribir_eventos(directorio, nombre, eventos):
    archivo = Path(directorio) / f"{nombre}.json"
    with archivo.open("w", encoding="utf-8") as f:
        for evento in eventos:
            fila = dict(evento)
            if isinstance(fila.get("timestamp"), datetime):
                fila["timestamp"] = fila["timestamp"].strftime("%Y-%m-%dT%H:%M:%S")
            f.write(json.dumps(fila, ensure_ascii=False) + "\n")
    print("Archivo creado:", archivo)

def evento(event_id, ts):
    return {"event_id": event_id, "timestamp": ts, "org_id": "org_test",
            "service": "compute", "metric": "requests", "value": "1", "unit": "count",
            "cost_usd_increment": 0.1, "schema_version": 2}

tardio_stream = (
    spark.readStream
    .schema(schema_events)
    .json(str(tardio_input))
    .withWatermark("timestamp", "10 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
)

for q in spark.streams.active:
    if q.name == "prueba_tardio":
        q.stop()

query_tardio = (
    tardio_stream.writeStream
    .format("memory")
    .queryName("prueba_tardio")
    .outputMode("append")
    .option("checkpointLocation", str(tardio_checkpoint))
    .trigger(processingTime="2 seconds")
    .start()
)

t0 = datetime(2025, 8, 1, 10, 0)

# Primer lote
escribir_eventos(tardio_input, "lote_001", [
    evento("t1", t0),
    evento("t2", t0 + timedelta(minutes=5)),
])
query_tardio.processAllAvailable()

# Segundo lote: hace avanzar el watermark a 10:20
escribir_eventos(tardio_input, "lote_002", [
    evento("t3", t0 + timedelta(minutes=30)),
])
query_tardio.processAllAvailable()

# Tercer lote: evento muy tardío (10:02, anterior al watermark)
escribir_eventos(tardio_input, "lote_003_tardio", [
    evento("t4", t0 + timedelta(minutes=2)),
])
query_tardio.processAllAvailable()

spark.sql("""
SELECT event_id, timestamp
FROM prueba_tardio
ORDER BY timestamp
""").show(truncate=False)                                    # t1, t2, t3 (sin t4)

p = query_tardio.lastProgress
print("Event time / watermark:", p.get("eventTime", {}))
print("Descartados por watermark:", p["stateOperators"][0]["numRowsDroppedByWatermark"])   # 1

query_tardio.stop()

Archivo creado: /content/cloud-provider-analytics/data/test_tardio/in/lote_001.json
Archivo creado: /content/cloud-provider-analytics/data/test_tardio/in/lote_002.json
Archivo creado: /content/cloud-provider-analytics/data/test_tardio/in/lote_003_tardio.json
+--------+-------------------+
|event_id|timestamp          |
+--------+-------------------+
|t1      |2025-08-01 10:00:00|
|t2      |2025-08-01 10:05:00|
|t3      |2025-08-01 10:30:00|
+--------+-------------------+

Event time / watermark: {'avg': '2025-08-01T10:02:00.000Z', 'max': '2025-08-01T10:02:00.000Z', 'min': '2025-08-01T10:02:00.000Z', 'watermark': '2025-08-01T10:20:00.000Z'}
Descartados por watermark: 1


In [61]:
# El keyspace cloud_provider se crea desde el portal de AstraDB (base -> Add Keyspace),
# porque Astra no admite CREATE KEYSPACE por CQL.

# prueba de conexión a Astra (Cassandra)
!pip -q install cassandra-driver

import glob
from google.colab import userdata
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

bundle = glob.glob("/content/secure-connect*.zip")[0]
# Removemos posibles comillas dobles que tenga el token guardado en Secrets
token = userdata.get("ASTRA_TOKEN").strip('"')

cluster = Cluster(
    cloud={"secure_connect_bundle": bundle},
    auth_provider=PlainTextAuthProvider("token", token),
)
session = cluster.connect("cloud_provider")

fila = session.execute("SELECT release_version FROM system.local").one()
print("Conectado. Versión de Cassandra:", fila.release_version)
print("Keyspace en uso:", session.keyspace)

Conectado. Versión de Cassandra: 4.0.11.0-6819aa9c7f6c
Keyspace en uso: cloud_provider


In [62]:
# tabla 1 en Cassandra + carga desde Gold
# Clave: partición por org_id y orden por (usage_date, service), porque las tres
# áreas consultan por organización y período (ver evidence/02_decisiones/decisiones.md).
from cassandra.concurrent import execute_concurrent_with_args

GOLD_PATH = os.path.join(REPO, "data/gold/org_daily_usage_by_service")

# La tabla se crea solo si no existe. Re-ejecutar el notebook no duplica filas:
# el INSERT en Cassandra es un upsert por clave primaria (org_id, usage_date, service).
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_provider.org_daily_usage_by_service (
    org_id text,
    usage_date date,
    service text,
    daily_cost_usd double,
    requests bigint,
    cpu_hours double,
    storage_gb_hours double,
    genai_tokens bigint,
    carbon_kg double,
    n_events bigint,
    PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date ASC, service ASC)
""")

filas = spark.read.parquet(GOLD_PATH).collect()

insert = session.prepare("""
INSERT INTO cloud_provider.org_daily_usage_by_service
(org_id, usage_date, service, daily_cost_usd, requests, cpu_hours,
 storage_gb_hours, genai_tokens, carbon_kg, n_events)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

# requests es double en Gold pero todos sus valores son enteros (verificado:
# 0 filas con decimales y suma igual en Gold y Cassandra), por eso va como bigint.
args = [(r["org_id"], r["usage_date"], r["service"], r["daily_cost_usd"],
         None if r["requests"] is None else int(r["requests"]),
         r["cpu_hours"], r["storage_gb_hours"], r["genai_tokens"],
         r["carbon_kg"], r["n_events"]) for r in filas]

resultados = execute_concurrent_with_args(session, insert, args, concurrency=50)
errores = [r for ok, r in resultados if not ok]

en_cassandra = session.execute(
    "SELECT COUNT(*) FROM cloud_provider.org_daily_usage_by_service").one()[0]
print("Filas en Gold:", len(filas))
print("Filas en Cassandra:", en_cassandra)
print("Errores de escritura:", len(errores))

Filas en Gold: 11050
Filas en Cassandra: 11050
Errores de escritura: 0


In [63]:
from pyspark.sql import functions as F

g = spark.read.parquet(GOLD_PATH)
print("Gold: filas con requests no entero:",
      g.filter(F.col("requests") != F.floor("requests")).count())
print("Gold: suma de requests:", g.agg(F.sum("requests")).first()[0])

c = session.execute("SELECT SUM(requests) FROM cloud_provider.org_daily_usage_by_service").one()[0]
print("Cassandra: suma de requests:", c)

Gold: filas con requests no entero: 0
Gold: suma de requests: 2281020.0


Cassandra: suma de requests: 2281020


In [64]:
# consulta CQL 1 — costos y requests diarios de una organización en un rango
q1 = """
SELECT usage_date, service, daily_cost_usd, requests
FROM cloud_provider.org_daily_usage_by_service
WHERE org_id = 'org_dbdw2pcn'
  AND usage_date >= '2025-07-01'
  AND usage_date <= '2025-07-14'
"""
r1 = list(session.execute(q1))
for f in r1:
    print(f.usage_date, f.service, f.daily_cost_usd, f.requests)
print("Filas:", len(r1), "| costo total:", round(sum(f.daily_cost_usd for f in r1), 4))

2025-07-03 database 20.463 383
2025-07-04 database 9.1282 120
2025-07-05 compute 10.5084 110
2025-07-05 database 20.5993 367
2025-07-05 networking 1.806 125
2025-07-06 compute 26.692 299
2025-07-06 database 15.5938 328
2025-07-06 networking 0.3233 None
2025-07-07 compute 18.8965 267
2025-07-07 database 1.7029 None
2025-07-07 networking 2.6985 240
2025-07-08 compute 20.5009 251
2025-07-08 database 6.8366 110
2025-07-08 networking 1.3052 116
2025-07-09 compute 20.7514 257
2025-07-09 database 10.7993 232
2025-07-09 networking 1.5492 135
2025-07-10 compute 21.7875 277
2025-07-10 database 11.9865 216
2025-07-10 networking 0.2637 None
2025-07-11 compute 19.1478 226
2025-07-11 database 17.7013 357
2025-07-11 networking 0.9433 102
2025-07-12 compute 1.6501 None
2025-07-12 database 16.9153 352
2025-07-12 networking 1.6244 205
2025-07-13 compute 39.8767 459
2025-07-13 database 8.9409 134
2025-07-13 networking 0.0663 None
2025-07-14 compute 21.7431 245
2025-07-14 database 8.6132 146
Filas: 31 | c

In [65]:
# consulta CQL 2 — servicios que más costaron en los últimos 14 días de una organización
from collections import defaultdict

q2 = """
SELECT usage_date, service, daily_cost_usd
FROM cloud_provider.org_daily_usage_by_service
WHERE org_id = 'org_dbdw2pcn'
  AND usage_date >= '2025-08-18'
  AND usage_date <= '2025-08-31'
"""
r2 = list(session.execute(q2))

# Cassandra devuelve las filas de la partición de la organización en ese rango;
# la suma por servicio y el orden se hacen acá, porque CQL no agrupa por una
# columna que está después de usage_date en la clave.
costo_por_servicio = defaultdict(float)
for f in r2:
    costo_por_servicio[f.service] += f.daily_cost_usd

print("Filas leídas de Cassandra:", len(r2))
for servicio, costo in sorted(costo_por_servicio.items(), key=lambda x: -x[1]):
    print(servicio, round(costo, 2))

Filas leídas de Cassandra: 31
compute 278.1
database 133.29
networking 8.39
